In [ ]:
%load_ext autoreload
%autoreload 2
%aimport -tqdm, -matplotlib, -torch, -torchinfo, -numpy, -timm, -cv2, -rich, -torchvision
# !export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module=".*albumentations.*")

import os
import sys
from tqdm.auto import tqdm as _tqdm
import tqdm as tqdm_module

def tqdm(*args, **kwargs):
    try:
        disable = globals().get("is_papermill", False) or bool(int(os.environ['NO_PROGRESSBARS']))
    except KeyError:
        disable = not sys.stdout.isatty()
    except (ValueError, TypeError):
        disable = False
    kwargs.setdefault('disable', disable)
    return _tqdm(*args, **kwargs)
tqdm_module.tqdm = tqdm

import torch
import matplotlib.pyplot as plt
import numpy as np
from numpy.typing import NDArray

try:
    notebook_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    notebook_dir = os.getcwd()
sys.path.append(os.path.abspath(os.path.join(notebook_dir, '..')))

import utils
from pathlib import Path
from utils.torch.datasets import QueriedFaceDataset
import model as my_model
%matplotlib inline

default_datasets_dir = "/home/dogsch/dev/datasets"
default_checkpoint_path = "/home/dogsch/dev/pa-face-landmark-tracking/results/baseline-s1.pth"

dataset_dir = Path(globals().get("dataset_dir", default_datasets_dir))
checkpoint_path = globals().get("checkpoint_path", default_checkpoint_path)
out_dir = Path(globals().get("out_dir", "."))
out_suffix = globals().get("out_suffix", "")

print(f"dataset_dir={dataset_dir}")
print(f"checkpoint_path={checkpoint_path}")

## Load Datasets

In [ ]:
import data as my_data
from utils.datasets import CanonicalLandmarks, VideoDataset
from utils.torch.datasets import sample_random_clips
from data import queries_68_ibug, queries_70_synth, queries_98_wflw, queries_opt, face_mesh, DatasetName

QueriedFaceDataset.unregister_all()
datasets = my_data.Datasets(dataset_dir)

print(datasets)

## Load Checkpoint

In [ ]:
import utils.torch
from model.utils import LandmarkPrediction
from utils.torch.datasets import QueriedFaceDataset

model = my_model.QLOT(feature_extractor_pretrained=False)
model_queries = my_data.make_query_points()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_queries(state_dict: dict):
    model_queries.load_state_dict(state_dict)

extra_save_args = {
    "query_points": model_queries,
    "model_func": my_model.QLOT.translate_weights
}

global_step, config = utils.torch.misc.load(
   checkpoint_path,
   model,
   None,
   None,
   strict=True,
   **extra_save_args
)

model.to(device)
model_queries.to(device)
del config.others["_best_checkpoints"]

config, global_step

In [ ]:
import profile_model
ITERATIONS = 4

profiler = profile_model.PyTorchProfiler(checkpoint_path, (1, 3, 224, 224), 98)
profiler.profile(print_summary=False, iterations=ITERATIONS)
profiler.profile(print_summary=True, iterations=1)

In [ ]:
from functools import partial
from eval import *

iod_indices_wflw = (60, 72)  # Indices for left and right eye corners in 98-point markup
eval_wflw = partial(eval_images, datasets=datasets, iod_indices=iod_indices_wflw, dataset_name=DatasetName.WFLW, device=device)

iod_indices_ibug = (36, 45)  # Indices for left and right eye corners in 68-point markup
eval_ibug = partial(eval_images, datasets=datasets, iod_indices=iod_indices_ibug, dataset_name=DatasetName.Ibug, device=device)

iod_indices_face_synth = (36, 45)  # Indices for left and right eye corners in 70-point markup
eval_face_synth = partial(eval_images, datasets=datasets, iod_indices=iod_indices_face_synth, dataset_name=DatasetName.FaceSynth, device=device)

## Eval WFLW

In [ ]:
import random

queries = model_queries.get(DatasetName.WFLW).to(device).unsqueeze(0)
model.eval()

idx = 1477 #random.randint(0, len(wflw_test_full))
print(f"idx = {idx}")
sample = datasets.wflw_test_full[idx]

img_raw = sample["image"]
img = sample["image"].unsqueeze(0).to(device)

preds: LandmarkPrediction = model(img, queries, iterations=3, store_similarity_maps=True, use_naive_correlation=True)
similarity_maps = model.prev_similarity_maps
assert similarity_maps is not None

plt.imshow(img_raw.permute(1, 2, 0).cpu().numpy().clip(0, 1))
plt.figure()
plt.imshow(similarity_maps[0][0, 35, 0].detach().cpu().numpy(), cmap="magma")
plt.figure()
plt.imshow(similarity_maps[1][0, 35, 0].detach().cpu().numpy(), cmap="magma")
plt.figure()
plt.imshow(similarity_maps[2][0, 35, 0].detach().cpu().numpy(), cmap="magma")

In [ ]:
if False:
    from pathlib import Path
    import matplotlib as mpl
    import numpy as np
    from PIL import Image

    out_dir = Path("/home/dogsch/dev/pa-face-landmark-tracking-paper/docs/thesis/figures")
    out_dir.mkdir(parents=True, exist_ok=True)

    # --- 1) Raw colour image, saved at its native (H, W) ---
    # img_raw is (C, H, W) and (after albumentations) already in [0, 1].
    raw = img_raw.detach().cpu().clamp(0, 1)
    raw_np = (raw.permute(1, 2, 0).numpy() * 255).round().astype(np.uint8)  # (H, W, 3)
    Image.fromarray(raw_np).save(out_dir / f"wflw_{idx:03d}_image.png")

    # --- 2-4) Similarity maps as magma-coloured RGB at their native (H, W) ---
    magma = mpl.colormaps["magma"]   # use this; `cm.get_cmap` is deprecated in mpl 3.7+

    def save_smap(t: torch.Tensor, path: Path) -> None:
        arr = t.detach().cpu().float().numpy()
        lo, hi = arr.min(), arr.max()
        arr = (arr - lo) / (hi - lo + 1e-12)        # normalise to [0, 1]
        rgb = (magma(arr)[..., :3] * 255).round().astype(np.uint8)  # drop alpha
        Image.fromarray(rgb).save(path)

    for i in (0, 1, 2):
        save_smap(similarity_maps[i][0, 35, 0], out_dir / f"wflw_{idx:03d}_sim_map_{i}.png")

In [ ]:
wflw_test_evals = eval_wflw(model, model_queries, split="full", batch_size=64, iterations=ITERATIONS)
wflw_blur_evals = eval_wflw(model, model_queries, split="blur", batch_size=64, iterations=ITERATIONS)
wflw_expression_evals = eval_wflw(model, model_queries, split="expression", batch_size=64, iterations=ITERATIONS)
wflw_illumination_evals = eval_wflw(model, model_queries, split="illumination", batch_size=64, iterations=ITERATIONS)
wflw_largepose_evals = eval_wflw(model, model_queries, split="largepose", batch_size=64, iterations=ITERATIONS)
wflw_makeup_evals = eval_wflw(model, model_queries, split="makeup", batch_size=64, iterations=ITERATIONS)
wflw_occlusion_evals = eval_wflw(model, model_queries, split="occlusion", batch_size=64, iterations=ITERATIONS)

wflw_evals = {
    "full": wflw_test_evals,
    "blur": wflw_blur_evals,
    "expression": wflw_expression_evals,
    "illumination": wflw_illumination_evals,
    "largepose": wflw_largepose_evals,
    "makeup": wflw_makeup_evals,
    "occlusion": wflw_occlusion_evals,
}

plt.figure(figsize=(8, 6))
plt.title("WFLW Cumulative Error Distribution")
for split_name, evals in wflw_evals.items():
    evals.summary()
    thr, ced = evals.ced_values(threshold=10.0, norm_type="iod")
    plt.plot(thr, ced, label=f"{split_name}")
plt.xlabel("NME Threshold (%)")
plt.ylabel("Percentage of Images (%)")
plt.xlim([0, 10])
plt.ylim([0, 100])
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
wflw_indices = [[2383, 904, 1218], [68, 748, 161], [27, 119, 160], [682, 193, 365], [205, 42, 285], [20, 63, 151], [438, 364, 45]]
for (split_name, evals), indices in zip(wflw_evals.items(), wflw_indices):
    evals.display(
        title=f"WFLW {split_name}",
        indices=indices,
        resolution=900,
        width=2,
        radius=6,
    )

## 300W / Ibug

In [ ]:
from utils.torch.datasets import QueriedFaceDataset

ibug_test_sets = {
    "common": datasets.ibug_test_common,
    "challenging": datasets.ibug_test_challenging,
    "outdoor": datasets.ibug_test_outdoor,
    "indoor": datasets.ibug_test_indoor,
}

In [ ]:
ibug_eval_common = eval_ibug(model, model_queries, split="common", iterations=ITERATIONS, batch_size=64)
ibug_eval_challenging = eval_ibug(model, model_queries, split="challenging", iterations=ITERATIONS, batch_size=64)
ibug_eval_outdoor = eval_ibug(model, model_queries, split="outdoor", iterations=ITERATIONS, batch_size=64)
ibug_eval_indoor = eval_ibug(model, model_queries, split="indoor", iterations=ITERATIONS, batch_size=64)

ibug_evals = {
    "common": ibug_eval_common,
    "challenging": ibug_eval_challenging,
    "outdoor": ibug_eval_outdoor,
    "indoor": ibug_eval_indoor,
}

plt.figure(figsize=(8, 6))
plt.title("300W Cumulative Error Distribution")
for split_name, evals in ibug_evals.items():
    evals.summary()
    thr, ced = evals.ced_values(norm_type="iod")
    plt.plot(thr, ced, label=f"{split_name}")
plt.xlabel("NME Threshold (%)")
plt.ylabel("Percentage of Images (%)")
plt.xlim([0, 10])
plt.ylim([0, 100])
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
ibug_indices = [[198, 121, 355], [70, 114, 3], [280, 170, 56], [239, 294, 276]]
for (name, evals), indices in zip(ibug_evals.items(), ibug_indices):
    evals.display(
        title=f"300W {name}",
        indices=indices,
        resolution=900,
        width=2,
        radius=6,
    )

# Face Synthetics

In [ ]:
face_synth_eval = eval_face_synth(model, model_queries, iterations=ITERATIONS, batch_size=64)

face_synth_evals = {
    "test": face_synth_eval
}

plt.figure(figsize=(8, 6))
plt.title("Face Synthetics Cumulative Error Distribution")
for split_name, evals in face_synth_evals.items():
    evals.summary()
    thr, ced = evals.ced_values(norm_type="iod")
    plt.plot(thr, ced)
plt.xlabel("NME Threshold (%)")
plt.ylabel("Percentage of Images (%)")
plt.xlim([0, 10])
plt.ylim([0, 100])
plt.grid(True)
plt.show()

In [ ]:
face_synth_indices = [[941, 2119, 366, 1965, 1506, 831, 1677, 597]]
for (name, evals), indices in zip(face_synth_evals.items(), face_synth_indices):
    evals.display(
        title=f"Face Synthetics {name}",
        indices=indices,
        resolution=900,
        width=2,
        radius=6,
        row_cols=(2, 4)
    ).show()

## Jitter on WFLW-V

In [ ]:
from utils.datasets.video import WFLW_V
from utils.datasets.image import WFLW_V_Frames

wflw_v_test = datasets.wflw_v_test

assert isinstance(wflw_v_test.dataset, WFLW_V_Frames)
print(f"clip_len={wflw_v_test.dataset.clip_len} frames")

In [ ]:
assert isinstance(wflw_v_test.dataset, WFLW_V_Frames)
wflw_v_videos: WFLW_V = wflw_v_test.dataset.videos
wflw_v_test_clip_names: list[str] = wflw_v_test.dataset.ordered_video_names

clips_hard_indices = {i for i, n in enumerate(wflw_v_test_clip_names) if n in wflw_v_videos.hard_video_ids}
clips_easy_indices = {i for i, n in enumerate(wflw_v_test_clip_names) if n in wflw_v_videos.easy_video_ids}

print(f"total_clips={len(wflw_v_test)}, hard_clips={len(clips_hard_indices)}, easy_clips={len(clips_easy_indices)}")

In [ ]:
if False:
    fig, ax = plt.subplots(1, 4, figsize=(24,6))
    video_idx = np.random.randint(0, len(wflw_v_test))

    sample = wflw_v_test[video_idx]
    sample = QueriedFaceDataset.wrap(sample)[30:34]

    for i, ax in enumerate(ax.flatten()):
        curr_sample = sample[i]
        ax.imshow(curr_sample.images.cpu().permute(1, 2, 0).numpy().clip(0, 1))
        ax.scatter(
            curr_sample.labels[:, 0].cpu().numpy(),
            curr_sample.labels[:, 1].cpu().numpy(),
            s=12,
            c='g'
        )
        del curr_sample
    fig.suptitle(f"WFLW-V {wflw_v_test_clip_names[video_idx]} ({"easy" if video_idx in clips_easy_indices else "hard"})", fontsize=16)
    fig.tight_layout()
    del sample

In [ ]:
from model.utils import NUM_PREDS_COORDS, NUM_PREDS_COV_PARAMS
BATCH_SIZE = 16
dataloader = utils.torch.datasets.DataLoader(wflw_v_test, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
queries = model_queries.get(DatasetName.WFLW_V).to(device).unsqueeze(0)
model.eval()

assert isinstance(wflw_v_test.dataset, WFLW_V_Frames)
wflw_v_labels = torch.zeros(len(wflw_v_test), wflw_v_test.dataset.clip_len, wflw_v_test.dataset.nlandmarks, NUM_PREDS_COORDS)
wflw_v_preds = torch.zeros(len(wflw_v_test), wflw_v_test.dataset.clip_len, wflw_v_test.dataset.nlandmarks, NUM_PREDS_COORDS)
wflw_v_cov = torch.zeros(len(wflw_v_test), wflw_v_test.dataset.clip_len, wflw_v_test.dataset.nlandmarks, NUM_PREDS_COV_PARAMS)
wflw_v_nme = torch.zeros(len(wflw_v_test), wflw_v_test.dataset.clip_len)
wflw_v_is_hard = torch.zeros(len(wflw_v_test), dtype=torch.bool)

with torch.no_grad():
    prev_hidden_state = None
    prev_predictions = None
    for i, batch in enumerate(tqdm(dataloader)):
        prev_hidden_state = None
        prev_predictions = None

        batch = QueriedFaceDataset.wrap(batch, device=device)
        clips_slice = slice(i * BATCH_SIZE, i * BATCH_SIZE + batch.batch_size)

        for frame_idx in range(batch.clip_len):
            frame_batch = batch[:, frame_idx]
            curr_queries = queries.expand(batch.batch_size, -1, -1)

            preds: LandmarkPrediction
            preds, hidden_state = model(
                frame_batch.images,
                curr_queries,
                iterations=ITERATIONS if frame_idx == 0 else 1,
                return_hidden_state=True,
                prefill_hidden_state=prev_hidden_state,
                prefill_starting_landmarks=prev_predictions,
            )

            prev_hidden_state = hidden_state
            prev_predictions = preds

            xy = preds.mean.cpu()  # (batch_size, num_queries, 2)
            labels: torch.Tensor = frame_batch.labels.cpu()

            errors = (xy - labels).square().sum(dim=-1).sqrt()  # (batch_size, num_queries)

            # Normalize by face size
            labels_bbox_max = torch.max(labels, dim=1).values  # (batch_size, 2)
            labels_bbox_min = torch.min(labels, dim=1).values  # (batch_size, 2)
            face_sizes = (labels_bbox_max - labels_bbox_min).prod(dim=-1).sqrt()  # (batch_size, 2)
            face_sizes = face_sizes.unsqueeze(-1)  # (batch_size, 1)
            nme = errors / face_sizes  # (batch_size, num_queries)

            wflw_v_labels[clips_slice, frame_idx] = labels
            wflw_v_preds[clips_slice, frame_idx] = xy
            try:
                wflw_v_cov[clips_slice, frame_idx] = preds.cov.params.cpu()
            except:
                pass
            wflw_v_nme[clips_slice, frame_idx] = nme.mean(dim=-1)  # Average NME across landmarks
            wflw_v_is_hard[clips_slice] = torch.tensor(
                [video_idx in clips_hard_indices for video_idx in range(clips_slice.start, clips_slice.stop)], dtype=torch.bool
            )

In [ ]:
fig, ax = plt.subplots(3, 4, figsize=(24,18))
video_idx = 120 #np.random.randint(0, len(wflw_v_test))

sample = wflw_v_test[video_idx]

clip_start_idx = 30
CLIP_LEN = 12
CLIP_STEP = 1
sample = QueriedFaceDataset.wrap(sample)[clip_start_idx:clip_start_idx+CLIP_STEP*CLIP_LEN:CLIP_STEP]


for i, ax in enumerate(ax.flatten()):
    real_i = i * CLIP_STEP
    curr_sample = sample[i]

    all_xy = [
        curr_sample.labels.cpu(),  # Ground Truth,
        wflw_v_preds[video_idx, clip_start_idx + real_i].cpu(),  # Predicted
    ]
    try:
        from model import LowRankCov2D
        cov = LowRankCov2D(wflw_v_cov[video_idx, clip_start_idx + real_i]).as_cov2d_params().cpu()  # (num_queries, 3)
        all_cov = [
            torch.zeros_like(cov),  # Ground Truth (not available)
            cov,  # Predicted
        ]
    except:
        all_cov = None

    img = draw_keypoints(
        curr_sample.images,
        all_xy,
        all_cov,
        colors=["green", "red"],
        probability_threshold=0.95,
        scale_to=900,
        radius=6,
        width=2,
    )
    ax.imshow(img.cpu().permute(1, 2, 0).contiguous().numpy())
    del curr_sample
fig.suptitle(f"WFLW-V {wflw_v_test_clip_names[video_idx]} ({"easy" if video_idx in clips_easy_indices else "hard"})", fontsize=16)
assert (video_idx in clips_hard_indices) == wflw_v_is_hard[video_idx].item()

fig.tight_layout(rect=[0, 0, 1, 0.98])  # type: ignore
del sample

In [ ]:
import matplotlib.colors
from matplotlib.patches import Ellipse
wflw_v_sample_errors = (wflw_v_preds[video_idx] - wflw_v_labels[video_idx])  # (clip_len, num_queries, 2)
wflw_v_sample_error_steps = wflw_v_sample_errors.diff(dim=0)  # (clip_len-1, num_queries, 2)
x_min, y_min = wflw_v_sample_errors.min(dim=0).values.min(dim=0).values.numpy()
x_max, y_max = wflw_v_sample_errors.max(dim=0).values.max(dim=0).values.numpy()
x_mean, y_mean = wflw_v_sample_errors.mean(dim=(0, 1)).numpy()
x_std, y_std = wflw_v_sample_errors.std(dim=(0, 1)).numpy()

print(f"Error Range x_min={x_min:.1f} px, x_max={x_max:.1f} px, y_min={y_min:.1f} px, y_max={y_max:.1f} px")
print(f"Mean Error x_mean={x_mean:.1f} px, y_mean={y_mean:.1f} px")
print(f"Std Error x_std={x_std:.1f} px, y_std={y_std:.1f} px")

fig, ax = plt.subplots(14, 7, figsize=(20, 40))
axes: list[plt.Axes] = ax.flatten().tolist()
lmks = my_data.queries_98_wflw
p = 0.95
k = np.sqrt(-2 * np.log(1 - p))

for lmk_i, (ax, error_steps, errors) in enumerate(zip(axes, wflw_v_sample_error_steps.permute(1, 0, 2), wflw_v_sample_errors.permute(1, 0, 2))):
    ax.set_title(f"Landmark {lmk_i} ({lmks.indices.group_name(lmk_i)})")
    std_xy = errors.std(dim=0).cpu().numpy()  # (2,)
    mean_xy = errors.mean(dim=0).cpu().numpy()  # (2,)

    # Full covariance handles correlation (rotation), unlike per-axis std
    cov = np.cov(errors.cpu().numpy().T)  # (2, 2)
    eigvals, eigvecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(eigvecs[1, 1], eigvecs[0, 1]))  # largest eigenvector first
    ax.add_patch(Ellipse(
        (mean_xy[0], mean_xy[1]),
        2 * k * np.sqrt(eigvals[1]),
        2 * k * np.sqrt(eigvals[0]),
        angle=angle,
        fill=True, color="gray", alpha=0.2,
    ))

    n_steps = error_steps.shape[0]
    h_es = ax.quiver(
        wflw_v_sample_errors[:-1, lmk_i, 0].cpu().numpy(),
        wflw_v_sample_errors[:-1, lmk_i, 1].cpu().numpy(),
        error_steps[:, 0].cpu().numpy(),
        error_steps[:, 1].cpu().numpy(),
        np.arange(n_steps),                          # color value = step index
        cmap = matplotlib.colors.LinearSegmentedColormap.from_list(
            "magma_r_dark", plt.get_cmap("magma_r")(np.linspace(0.05, 0.75, n_steps))
        ),
        norm=matplotlib.colors.Normalize(0, max(n_steps - 1, 1)),
        angles='xy',
        scale_units='xy',
        scale=1,
        label="Error Steps",
    )
    h_me = ax.scatter(mean_xy[0], mean_xy[1], color="red", s=20, label="Mean Error")
    h_gt = ax.scatter(0, 0, color="green", s=20, label="Ground Truth")
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

    nme_iod_elem = ax.plot([], [], " ", label=f"$\\sigma=${np.sqrt((std_xy ** 2).sum()):.1f} px")
    nme_s_elem = ax.plot([], [], " ", label=f"$\\mu=${np.sqrt((mean_xy ** 2).sum()):.1f} px")
    ax.legend(handles=[*nme_iod_elem, *nme_s_elem], handletextpad=0.0, handlelength=0, fontsize=8)

fig.suptitle(f"WFLW-V {wflw_v_test_clip_names[video_idx]} Landmark Error Steps", fontsize=16)
fig.legend(handles=[h_me, h_gt], loc="upper right", ncols=3, bbox_to_anchor=(0.995, 0.983))  # type: ignore
fig.tight_layout(h_pad=0, rect=[0, 0, 1, 0.985])  # type: ignore
fig.colorbar(h_es, ax=axes, label="Frame", fraction=0.01, pad=0.01)  # type: ignore
fig.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6))

wflw_v_sample_error_means = wflw_v_sample_errors.mean(dim=1).numpy()
wflw_v_sample_error_stds = wflw_v_sample_errors.std(dim=1).numpy()

for i in range(wflw_v_sample_errors.shape[-1]):
    var = "x" if i == 0 else "y"
    color = "tab:blue" if i == 0 else "tab:orange"
    ax.fill_between(
        np.arange(len(wflw_v_sample_error_means)),
        wflw_v_sample_error_means[..., i] - wflw_v_sample_error_stds[..., i],
        wflw_v_sample_error_means[..., i] + wflw_v_sample_error_stds[..., i],
        color=color,
        alpha=0.2,
        label=f"{var} Std Error"
    )
    ax.plot(
        np.arange(len(wflw_v_sample_error_means)),
        wflw_v_sample_error_means[..., i],
        color=color,
        label=f"{var} Mean Error"
    )
ax.hlines(
    y=0,
    xmin=0,
    xmax=len(wflw_v_sample_error_means) - 1,
    color="green",
    linestyle="--",
    label="Ground Truth"
)
ax.set_xlabel("Frame Index")
ax.set_ylabel("Mean Landmark Error (px)")
ax.set_title(f"WFLW-V {wflw_v_test_clip_names[video_idx]} Mean Landmark Error Over Time")
ax.legend()
fig.show()

In [ ]:
from utils.torch.misc import calc_nmf

wflw_v_nmf = calc_nmf(wflw_v_preds, wflw_v_labels)

print(f"---- WFLW_V ----")
print(f"WFLW-V NMF = {wflw_v_nmf:.4f}")

nme_per_clip = wflw_v_nme.mean(dim=-1).numpy() * 100.0  # Convert to percentage)
worst_nmes = nme_per_clip[np.argsort(-nme_per_clip)[:10]]

print(f"Worst NMEs = [{', '.join([f'{nme:.2f}%' for nme in worst_nmes])}]")
wflw_v_nme_size = np.mean(nme_per_clip)
print(f"NME Size = {wflw_v_nme_size:.2f}%")

In [ ]:
from utils.torch.misc import navar_pos_all

wflw_v_navar_pos = navar_pos_all(wflw_v_preds, wflw_v_labels).numpy()
m_vals = np.arange(1, wflw_v_navar_pos.shape[0] + 1)

In [ ]:
for lmk_i in range(wflw_v_navar_pos.shape[-1]):
    plt.plot(m_vals, np.sqrt(wflw_v_navar_pos[:, :, lmk_i]).mean(axis=1), label=f"Lmk {lmk_i}")
plt.xlabel("Block size (m)")
plt.ylabel("NADEV")
plt.title("NAVAR vs Block Size for Each Landmark")

In [ ]:
plt.loglog(m_vals, np.sqrt(wflw_v_navar_pos.mean(axis=-1)).mean(axis=-1), label=f"RMS across landmarks")
plt.xlabel("Block size (m)")
plt.ylabel("NADEV")
plt.title("Normalized Allan Deviation (NADEV) vs Block Size")

## Save Results

In [ ]:
import pickle
results = {
    "name": config.run,
    "step": global_step,
    "wflw": {
        k: v.as_dict() for k, v in wflw_evals.items()
    },
    "ibug": {
        k: v.as_dict() for k, v in ibug_evals.items()
    },
    "face_synth": {
        k: v.as_dict() for k, v in face_synth_evals.items()
    },
    "wflw_v": {
        "clip_names": wflw_v_test_clip_names,
        "labels": wflw_v_labels.numpy(),  # (num_clips, clip_len, num_landmarks, 2)
        "preds": wflw_v_preds.numpy(),  # (num_clips, clip_len, num_landmarks, 2)
        "cov": wflw_v_cov.numpy(),  # (num_clips, clip_len, num_landmarks, 3) as (log std_x, log std_y, atanh(corr))
        "nmes_size": wflw_v_nme.numpy(),  # (num_clips, clip_len)
        "is_hard": wflw_v_is_hard.numpy(),  # (num_clips,)
        "nmf": float(wflw_v_nmf),
        "nme_size": float(wflw_v_nme_size),
        "navar_pos": wflw_v_navar_pos,  # (m = 1...120, num_clips, num_landmarks)
    }
}
with open(out_dir / f"results_{config.run}{out_suffix}.pkl", "wb") as f:
    pickle.dump(results, f)